In [1]:
import forallpeople as si

# Load SI base + derived units
si.environment("structural", top_level=True)

# Define a speed in SI
speed_si = 10 * kN
impl = 1*m**3
test = speed_si/impl.to(si.m**2)

# Convert to mph (imperial)
#speed_mph = speed_si.to(mph)

# Show both
print("Speed in SI:", test)
#print("Speed in Imperial:", speed_mph)


Speed in SI: 10.000 kN/m³


c:\Users\datho\PythonProjects\PakCalcs\venv\Lib\site-packages\forallpeople\__init__.py:177: UserWarning: No unit defined for '1.000 m²' on 1.000 m³.
  warnings.warn(f"No unit defined for '{unit_name}' on {self}.")


In [2]:
from handcalcs.decorator import handcalc
import handcalcs.render
import pandas as pd
import pint
ureg = pint.UnitRegistry()
ureg.formatter.default_format = "~P"
import forallpeople as si
si.environment('default', top_level=True)


a = 1*m**2
b = 99*kg
c = a/b
print(c)


0.010 kg⁻¹·m²


# Geometry

In [3]:
%%render 2
A_max = (10 * m**2) # The max permitted roof area 
# c_w = (2.5*ft).to(m) # Width of a sheet of Custom Orb cladding
c_w = (0.7*m) # Width of a sheet of Spandek cladding
door_w = (3.71 * m) # Width of the double sliding door opening

<IPython.core.display.Latex object>

In [4]:
nosheets = list(range(5,9)) # number of sheets being considered
full_cov_w = [n * c_w for n in nosheets] # full coverage widths for each number of sheets
length_max = [A_max / w for w in full_cov_w] # max lengths for each full coverage width
roof_edge_width = [(w - door_w)/2 for w in full_cov_w] # width of roof edge on either side of door
data = {
    "NO Sheets": nosheets,
    "Total Width": full_cov_w,
    "Max. Length": length_max,
    "Edge Coverage": roof_edge_width
    }
df = pd.DataFrame(data)
print(df)
row_index = 1
w_b = df.loc[row_index, "Total Width"]
w_c = df.loc[row_index, "Max. Length"]
print("For", df.loc[row_index, "NO Sheets"], "sheets:")
print("  Total Width =", w_b)
print("  Max. Length =", w_c)


   NO Sheets Total Width Max. Length Edge Coverage
0          5     3.500 m     2.857 m   -105.000 mm
1          6     4.200 m     2.381 m    245.000 mm
2          7     4.900 m     2.041 m    595.000 mm
3          8     5.600 m     1.786 m    945.000 mm
For 6 sheets:
  Total Width = 4.200 m
  Max. Length = 2.381 m


### Loads
---
##### Dead Loads

In [29]:
gamma_rs = 9.81*m/s**2*5.28*kg/m**2  # mass per unit area of roof sheeting
gamma_t = (8000 * N / m**3).to(N/m**3)  # unit weight of timber
print("gamma_rs =", gamma_t.to(N/m**3))

gamma_rs = 8000.000 kg·m⁻²·s⁻²


c:\Users\datho\PythonProjects\PakCalcs\venv\Lib\site-packages\forallpeople\__init__.py:177: UserWarning: No unit defined for '1.000 kg·m⁻²·s⁻²' on 8000.000 kg·m⁻²·s⁻².
  warnings.warn(f"No unit defined for '{unit_name}' on {self}.")


For the beam supporting the roof sheeting

In [6]:
%%render
w_load = w_c/2
w_rs = (w_load * gamma_rs).to(N/m)


<IPython.core.display.Latex object>

### Loads
---
##### Wind Loads
__Regional Wind Speeds__

Region  A5


In [7]:
%%render params 1
V_Rservice = 37*m/s
V_Rult = 45*m/s




<IPython.core.display.Latex object>

__Multipliers__

In [8]:
%%render params 2
M_d = 1.0   
M_c = 1.0
h = 5*m
M_zcat = 0.83
M_s = 1.0
M_t = 1.0

<IPython.core.display.Latex object>

In [9]:
%%render 2 short
V_sit_beta_s = V_Rservice * M_d * M_c * M_zcat * M_s * M_t
V_sit_beta_u = V_Rult * M_d * M_c * M_zcat * M_s * M_t
V_des_theta_s = V_sit_beta_s
V_des_theta_u = V_sit_beta_u
sigma_air = (1.2 * si.kg / si.m**3)
p_theta_s = 0.5 * sigma_air * V_des_theta_s**2
p_theta_u = 0.5 * sigma_air * V_des_theta_u**2


<IPython.core.display.Latex object>

In [10]:
%%render params
h
h_c = 3*m
heightratio = h_c/h
w_c

<IPython.core.display.Latex object>

For wind at 0 degrees

In [11]:
%%render long
# From AS1170.2:2002 Table D8 for hc/h = heightratio
h_conh_1 = 0.5
pos_coeff_1 = 0.5
h_conh_2 = 0.75
pso_coeff_2 = 0.4
neg_coeff_1 = -0.3
neg_coeff_2 = min((-0.3-0.2*h_c/w_c),-1.5)


<IPython.core.display.Latex object>

In [12]:
%%render long
posC_pn0 = (heightratio-h_conh_1)/(h_conh_2 - h_conh_1)*(pso_coeff_2 - pos_coeff_1) + pos_coeff_1
negC_pn0 = (heightratio-h_conh_1)/(h_conh_2 - h_conh_1)*(neg_coeff_2 - neg_coeff_1) + neg_coeff_1


<IPython.core.display.Latex object>

In [13]:
%%render params
posC_pn0
negC_pn0

<IPython.core.display.Latex object>

For wind at 90° and 270°

In [14]:
%%render
width_d = w_b
canopy_height_h = h_c
h_on_d = canopy_height_h/width_d

<IPython.core.display.Latex object>

In [15]:
%%render
# From AS1170.2:2002 Table D4(A) with alpha = 0°
posC_pn90 = 0.4
negC_pn90 = -0.4

<IPython.core.display.Latex object>

Local factor k<sub>l

Roof slope  < 10°

In [16]:
%%render
a = 0.20 * w_c
k_l = 3.0

<IPython.core.display.Latex object>

Peak pressure on roof cladding

In [17]:
%%render long
p_dessposc = p_theta_s*max(posC_pn0, posC_pn90)*k_l
p_dessnegc = p_theta_s*min(negC_pn0, negC_pn90)*k_l
p_desuposc = p_theta_u*max(posC_pn0, posC_pn90)*k_l
p_desunegc = p_theta_u*min(negC_pn0, negC_pn90)*k_l

<IPython.core.display.Latex object>

Adopt Lysaght Spandek 0.48 BMT with 3 fasteners per sheet